# A3 -- pre-flight (stage 0, before the HGCC array)

Shared stage 0 for **Reviewer #2, major point 9** (`plans/Round2_response_analysis_plan.md`
section A3). Nothing here needs a new detection; everything here is needed *by* one.

> "The ambient background is modeled as complete spatial randomness (CSR), yet ambient RNA from
> debris, dying cells, and extracellular vesicles is typically spatially structured rather than
> uniform, and is likely to be denser precisely where cells are denser, or in regions of severe AD
> pathology... A CSR-based threshold could therefore under-correct in such regions and inflate
> granule calls locally."

### Why this is its own notebook

`preflight/set0_genes.csv` tells `run_detection_sets.py` which genes Set 0 seeds on, so it has to
exist **before** the array is submitted -- while every other A3 notebook needs the array's output.
Separating the two stages is what lets each notebook run exactly once, top to bottom, with no
toggle to flip. Full sequence: the runbook in `README.md`.

### What it produces

| output | read by |
|---|---|
| `preflight/set0_genes.csv` | `run_detection_sets.py` (array tasks 0-1) -- **upload this to HGCC** |
| `preflight/csr_min_samples.csv` | A3a gate (a); `A3_figures.R` section 1 |
| `preflight/set2_diagnostics.csv` | quotable table -- the numbers A3b's placement argument rests on |
| `preflight/z_profile_<sample>.csv` | `A3_figures.R` section 1 -- the flagged z-coverage issue |

**Run from `R2_revision/ambient_controls/`.**


## 0. Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")

# No runtime gates. This notebook runs once, top to bottom, before the HGCC array -- there is
# nothing here to subsample and nothing to defer.
C.ensure_dirs()
PRE = C.PREFLIGHT_DIR

print("data      :", C.DATA_ROOT)
print("writing to:", PRE)
print("samples   :", C.SAMPLES)


## 1. Pre-flight

Three things, none of which need the new detections:

1. **The CSR table.** `code/3_detection.py:66,83` passes `minspl=3`, so `poisson_select()` -- the
   CSR background model the reviewer objects to -- never runs on real data (`alpha=10` and
   `cutoff_prob=0.95` are inert arguments). At `alpha=10` the rule would have been far *stricter*
   than what was used; at **alpha = 0.5 it returns exactly 3 for all 20 markers in both samples**,
   so `min_samples = 3` **is** the CSR rule at that alpha and no result changes. This table is the
   evidence for that sentence in the response, and A3a's gate (a) asserts on it.
2. **Set 0's gene list**, persisted so the choice is auditable and identical across both samples.
3. **Set-2 diagnostics** -- the numbers the later sections and A3b depend on: 2D coverage
   fraction, the `layer_z` distribution, and the per-seed-gene composition.

Section numbering is **global across the A3 notebooks** and maps 1:1 onto `A3_figures.R`'s
sections, so `A3a_three_sets.ipynb` starts at section 2.


In [ ]:
csr_frames, diag_rows = [], []
for sample in C.SAMPLES:
    tx = A3.load_transcripts(sample, columns=["target", "global_x", "global_y",
                                              "global_z"])
    csr_frames.append(A3.csr_table(sample, transcripts=tx))

    area = A3.tissue_area(tx)
    g = pd.read_parquet(C.mcdetect_granules_path(sample))
    proj = float((np.pi * g["sphere_r"] ** 2).sum())
    n_planes = int(g["layer_z"].nunique())
    diag_rows.append(dict(
        sample=sample, n_granules=len(g), tissue_area_um2=area,
        n_z_planes=n_planes,
        coverage_all_planes=proj / area,
        coverage_per_plane=proj / (area * max(n_planes, 1)),
        median_sphere_r=float(g["sphere_r"].median()),
        frac_nc_ratio_zero=float((g["nc_ratio"] == 0).mean()) if "nc_ratio" in g else np.nan,
    ))
    # z-plane occupancy, both transcripts and granules -- this is the KNOWN ISSUE flagged in
    # the README: the AD section thins with depth while WT is flat.
    # rename_axis on BOTH before concat: two differing index names ("z" vs "layer_z") make
    # pandas drop the name entirely, and the CSV column comes out as "index" -- which the R
    # panel reads as `aes(x = z)` and fails on.
    zt = tx.groupby("global_z").size().rename("n_tx").rename_axis("z")
    zg = g.groupby("layer_z").size().rename("n_granules").rename_axis("z")
    (pd.concat([zt, zg], axis=1).rename_axis("z").reset_index()
       .assign(sample=sample).to_csv(PRE / f"z_profile_{sample}.csv", index=False))
    del tx

csr = pd.concat(csr_frames, ignore_index=True)
csr.to_csv(PRE / "csr_min_samples.csv", index=False)
diag = pd.DataFrame(diag_rows)
diag.to_csv(PRE / "set2_diagnostics.csv", index=False)

equiv = csr[(csr["alpha"] == C.CSR_ALPHA_EQUIV) & (csr["gene_set"] == "marker")]
print(f"alpha = {C.CSR_ALPHA_EQUIV}: markers give min_samples "
      f"{sorted(equiv['min_samples'].unique())} "
      f"(expected [{C.CSR_EXPECTED_MIN_SAMPLES}])")
wide = (csr[csr["alpha"].isin([C.CSR_ALPHA_EQUIV, 10.0])]
        .pivot_table(index=["sample", "gene_set", "gene"], columns="alpha",
                     values="min_samples"))
display(wide.sort_values(max(C.CSR_ALPHA_SWEEP), ascending=False).head(25))
display(diag)

In [ ]:
# Set 0. Selected on WT and reused for AD ON PURPOSE -- a per-sample list would let the two
# arms of the WT/AD contrast seed on different genes.
set0 = A3.select_set0("WT")
set0.to_csv(PRE / "set0_genes.csv", index=False)
print(f"Set 0: {set0['set0_gene'].notna().sum()} genes matched")
print("median |log10 abundance gap| =", round(float(set0["log10_gap"].median()), 3))

# How far the match actually holds. The panel has no unannotated gene above ~700K transcripts, so
# the match is near-exact for the rarer markers and degrades badly for the most abundant ones.
# This is why A3a section 3 leads with the PER-MILLION-TRANSCRIPT RATE and treats the raw sphere
# count as secondary -- a count comparison would inherit exactly the abundance objection Set 0
# exists to remove.
ratio = set0["n_tx_marker"] / set0["n_tx_set0"]
print(f"matched within 2x : {int(ratio.between(0.5, 2).sum())}/{len(set0)} markers")
print(f"worst gap         : {set0.loc[ratio.idxmax(), 'marker']} "
      f"{int(set0.loc[ratio.idxmax(), 'n_tx_marker']):,} vs "
      f"{set0.loc[ratio.idxmax(), 'set0_gene']} "
      f"{int(set0.loc[ratio.idxmax(), 'n_tx_set0']):,} ({ratio.max():.1f}x)")
print(f"aggregate         : markers {int(set0['n_tx_marker'].sum()):,} tx vs set0 "
      f"{int(set0['n_tx_set0'].sum()):,} tx ({set0['n_tx_marker'].sum() / set0['n_tx_set0'].sum():.2f}x)")
display(set0)

# The NC policy travels with the outputs, so a table can always be traced to which list
# produced it.
A3.write_run_info(PRE, stage="preflight", csr_alpha_equiv=C.CSR_ALPHA_EQUIV,
                  n_set0=int(set0["set0_gene"].notna().sum()),
                  n_seed_markers=len(C.SYN_GENES),
                  nc_list_new_use=C.NC_LIST_NEW_USE,
                  nc_list_published=C.NC_LIST_PUBLISHED)
print("\n--> now submit the HGCC array:  bash slurm/submit.sh")

## Outputs

| file | contents |
|---|---|
| `preflight/csr_min_samples.csv` | what `poisson_select` would return, per gene, over the alpha sweep -- backs the CSR disclosure |
| `preflight/set0_genes.csv` | the abundance-matched neutral gene list, read by `run_detection_sets.py` |
| `preflight/set2_diagnostics.csv` | n, tissue area, 2D coverage per plane, median radius, `nc_ratio == 0` fraction |
| `preflight/z_profile_<sample>.csv` | transcripts and granules per z-plane -- the flagged z-coverage issue |
| `preflight/run_info.csv` | the NC-list policy in force, so any table can be traced to the list that produced it |

### Next

Upload `output/preflight/set0_genes.csv` to HGCC and submit the array -- `README.md`, runbook
steps 2-3.
